# Seminar 3 - Real-Time Non-Maximum Suppression with CUDA/Numba

**Group 11**  
Phung Quoc Tuan (19127616) · Le Quang Tan (22127378)

## Scope

This report evaluates class-aware greedy hard NMS for one image: the transparent NumPy CPU baseline, CUDA/Numba V1 (dense pairwise IoU relation), and CUDA/Numba V2 (SoA reads with packed 64-bit suppression masks). V3 Matrix NMS is a separate experiment; it is not a hard-NMS parity result and is excluded from the primary comparison.

In [1]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
REPO_ROOT = next(path for path in (start, *start.parents) if (path / 'src' / 'cpu_baseline.py').is_file())
sys.path.insert(0, str(REPO_ROOT / 'src'))
print(f'Repository root: {REPO_ROOT}')

Repository root: D:\Study\HCMUS-APP\cuda-nms-numba\.worktrees\seminar3-submission


In [2]:
import platform
import numpy as np
import numba
from numba import cuda

print(f'Python: {platform.python_version()}')
print(f'NumPy: {np.__version__}')
print(f'Numba: {numba.__version__}')
print(f'CUDA available: {cuda.is_available()}')
assert cuda.is_available(), 'CUDA is required to reproduce the GPU results.'
print(f'GPU: {cuda.get_current_device().name.decode() if isinstance(cuda.get_current_device().name, bytes) else cuda.get_current_device().name}')

Python: 3.11.9
NumPy: 1.26.4
Numba: 0.67.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti


## Method

Greedy hard NMS has a sequential keep/suppress dependency. The CPU baseline resolves it directly. V1 launches one CUDA thread per box pair to construct a dense IoU relation, then resolves the greedy decision on the host. V2 lays out coordinates in structure-of-arrays form, produces packed suppression masks on CUDA, then performs the same host greedy mask resolution. Class partitions are processed independently and final kept indices are deterministically re-sorted.

In [3]:
from common.candidates import load_synthetic_candidates
from cpu_baseline import run_cpu
from gpu_v1 import run_gpu_v1
from gpu_v2 import run_gpu_v2

boxes, scores, class_ids = load_synthetic_candidates(64, seed=2026)
cpu_keep = run_cpu(boxes, scores, class_ids)
v1_keep = run_gpu_v1(boxes, scores, class_ids)
v2_keep = run_gpu_v2(boxes, scores, class_ids)
assert np.array_equal(cpu_keep, v1_keep)
assert np.array_equal(cpu_keep, v2_keep)
print(f'Kept candidates: {len(cpu_keep)}')
print('GPU V1/V2 parity: PASS')

D:\Study\HCMUS-APP\cuda-nms-numba\.worktrees\seminar3-submission\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Kept candidates: 64
GPU V1/V2 parity: PASS


D:\Study\HCMUS-APP\cuda-nms-numba\.worktrees\seminar3-submission\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [4]:
import json

evidence = REPO_ROOT / 'submission' / 'seminar_3' / 'evidence'
sweep = json.loads((evidence / 'benchmark_v1_v2.json').read_text(encoding='utf-8'))
batch = json.loads((evidence / 'batch32_v2.json').read_text(encoding='utf-8'))
environment = (evidence / 'environment.txt').read_text(encoding='utf-8')
source_commit = next(line.split('=', 1)[1] for line in environment.splitlines() if line.startswith('source_commit='))
print(f'Evidence source commit: {source_commit}')
assert sweep['benchmark_scope'] == 'nms_only_synthetic'
assert batch['benchmark_scope'] == 'nms_only_synthetic'

Evidence source commit: 7ee76cd5f6e12b87ddee247d58c9fd6ac866245b


In [5]:
print('NMS-only synthetic timing (median; lower is better)')
print(f"{'N':>6} {'CPU ms':>10} {'V1 ms':>10} {'V2 ms':>10} {'V1 speedup':>12} {'V2 speedup':>12}")
for n in (100, 1000, 10000):
    result = sweep['results'][str(n)]
    cpu_ms = result['cpu']['median_seconds'] * 1_000
    v1_ms = result['v1']['median_seconds'] * 1_000
    v2_ms = result['v2']['median_seconds'] * 1_000
    print(f'{n:>6} {cpu_ms:>10.3f} {v1_ms:>10.3f} {v2_ms:>10.3f} {cpu_ms / v1_ms:>11.2f}x {cpu_ms / v2_ms:>11.2f}x')
batch_ms = batch['median_batch_seconds'] * 1_000
per_image_ms = batch['median_per_image_seconds'] * 1_000
target_status = 'MET' if batch_ms < 5 else 'MISSED'
print(f'Batch-32 median: {batch_ms:.3f} ms total; {per_image_ms:.3f} ms/image')
print(f'Batch-32 target status: {target_status} (<5 ms/batch)')

NMS-only synthetic timing (median; lower is better)
     N     CPU ms      V1 ms      V2 ms   V1 speedup   V2 speedup
   100      1.378      2.467      5.065        0.56x        0.27x
  1000     16.259      4.501      7.625        3.61x        2.13x
 10000    308.965    113.991     33.327        2.71x        9.27x
Batch-32 median: 947.180 ms total; 29.599 ms/image
Batch-32 target status: MISSED (<5 ms/batch)


## Limitations and interpretation

The timings are machine-specific NMS-only measurements on the recorded RTX 4060 Ti environment; they exclude model inference and image preprocessing. The host performs the final greedy resolver, so the pipeline is not fully device-resident. The optional YOLOv5 checkpoint is not in this repository, therefore no detector-inference result is claimed. The batch-32 target is evaluated from the saved evidence and is reported honestly above. V3 Matrix NMS is not used as a hard-NMS oracle or parity claim.

## Reproduction

From the repository root (Python 3.11):

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements-cuda13.txt
.\.venv\Scripts\python.exe -m pytest -q
.\.venv\Scripts\python.exe benchmarks\run_all.py --n 100 1000 10000 --versions cpu v1 v2 --warmup 2 --repeats 7 --seed 0 --json submission\seminar_3\evidence\benchmark_v1_v2.json
.\.venv\Scripts\python.exe benchmarks\run_v2_batch.py --batch-size 32 --n 10000 --warmup 2 --repeats 7 --seed 0 --json submission\seminar_3\evidence\batch32_v2.json
```

Then execute this notebook from the repository root.